# 08 · Daily out-of-sample comparison

**Purpose.** Compare the daily models day by day. Each model, fitted at an origin, predicts the daytime low of every day of the following year from that day's own calendar, rooftop photovoltaic (PV) capacity and weather, and the predictions are scored against what each day did.

**Why it was added.** The annual targets of notebook 06 give seven test points per model, which cannot settle a comparison. The daily predictions give about 2,400. This notebook was added **after the first scores had been seen**, so it is supplementary to the comparisons fixed in advance.

**What it does and does not measure.** Nothing is simulated and no past miss is borrowed: these are the fitted models' own predictions.

- **M4 and M5 are given each day's actual weather**, which is not known a year ahead. So M4 against M3 and M5 against M6 measure how much weather *explains* once it is known, not how much it *helps* a forecast made a year ahead.
- **M2, M3 and M6 use no weather.** M3 against M2 is therefore a fair one-year-ahead test of PV capacity against time, with the target year's PV capacity taken as known, as everywhere in this study.

**Approximate runtime.** Under 30 seconds.

**Paper section served.** IV-B Results: forecast comparison (supplementary).

**Inputs**

- `data/interim/network_30min.parquet` from notebook 01.
- `data/interim/pv_daily.parquet` and `weather_daily.parquet` from notebook 02.

**Paper artifacts**

- `results/tables/table2.csv`: Table II, the daily skill of M2 to M6 and the three pairs of models, over all valid days and over the lowest 10% of days.

**Supporting checks** (`results/checks`)

- `daily_skill.csv`: mean absolute error and mean error by model, all test years together. Table II shows the mean absolute error only; the mean error and the number of days are kept here.
- `daily_skill_by_year.csv`: mean absolute error and mean error by model and test year.
- `daily_pairs_by_year.csv`: the three pairs by test year.

**Data for later use** (`data/interim`)

- `daily_errors.parquet`: the miss of each model on each valid test day.

**Main process**

1. Set up.
2. The daily predictions and the days they are scored on.
3. Skill by model: all test years together.
4. Skill by model and test year.
5. The three pairs.
6. Table II.
7. Result.

**How to read the scores.**

- **Mean absolute error** (MW) is the average size of a model's daily miss; lower is better.
- **Mean error** (MW) is the average miss with its sign, actual minus predicted. Positive means the model predicted too low on average; negative, too high.
- Each is given over **all valid days** and over the **lowest days**: the 10% of each year's valid days with the lowest actual daytime low, which are the days that set the annual low.

## 1. Set up

In [1]:
import pandas as pd

from pynmdf import evaluate
from pynmdf.config import load_config

cfg = load_config()
interim, tables, checks = (cfg["dirs"][key] for key in ("interim", "tables", "checks"))
lowest_share = cfg["evaluation"]["lowest_days_share"]

pd.set_option("display.width", 220, "display.max_columns", 40)
pd.set_option("display.float_format", "{:,.1f}".format)
total = pd.read_parquet(interim / "network_30min.parquet")["total_mw"]
pv = pd.read_parquet(interim / "pv_daily.parquet")["pv_mw"]
weather = pd.read_parquet(interim / "weather_daily.parquet")

# The three pairs, as (model, reference). A negative change means the first misses by less.
PAIRS = {
    "Impact of PV in place of time (M3 against M2)": ("M3", "M2"),
    "Impact of weather, linear (M4 against M3)": ("M4", "M3"),
    "Impact of weather, XGBoost (M5 against M6)": ("M5", "M6"),
}


def side_by_side(table: pd.DataFrame) -> pd.DataFrame:
    """Put the two day sets of a table next to each other."""
    sets = {name: table.xs(name, level="day_set") for name in evaluate.DAY_SETS}
    return pd.concat(sets, axis=1)

## 2. The daily predictions and the days they are scored on

For each test year from 2019 to 2025, each of M2 to M6 is fitted on the days up to the end of the year before, exactly as in the backtest of notebook 05, and then predicts the daytime low of every day of the test year. This is the same fit the forecasts of notebook 05 come from; nothing is changed or tuned here.

A day is **valid** when its actual daytime low exists, which needs both of the day's trough windows (00:00 to 06:00 and 09:00 to 17:00) to be complete: the same rule that decides which days the models are fitted on. Every model is scored on the same days.

The table gives the number of valid days in each test year and how many of them count as its lowest days.

In [2]:
daily = evaluate.own_weather_predictions(total, pv, weather, cfg)
errors = evaluate.daily_errors(daily, "daytime", lowest_share)
errors.to_parquet(interim / "daily_errors.parquet")

counts = errors.groupby(["year", "model"]).agg(
    valid_days=("error", "size"), lowest_days=("lowest", "sum")
)
# Every model must be scored on the same days of each year.
assert (counts.groupby("year").nunique() == 1).all().all()
days = counts.xs("M2", level="model").copy()
days.loc["all years"] = days.sum()
days.T

year,2019,2020,2021,2022,2023,2024,2025,all years
valid_days,354,347,331,352,350,320,360,2414
lowest_days,35,34,33,35,35,32,36,240


## 3. Skill by model: all test years together

One row per model. The left half is over all valid days, the right half over the lowest days only.

The table is saved as a check. Table II shows the mean absolute error only, so this is where the mean error and the number of days are kept.

In [3]:
skill = evaluate.daily_skill(errors, ["model"])
skill.to_csv(checks / "daily_skill.csv")
side_by_side(skill)

all days                     lowest days                   
        mae_mw mean_error_mw  days      mae_mw mean_error_mw days
model                                                            
M2       274.1        -115.1  2414       292.4        -286.4  240
M3       260.3         -13.1  2414       211.7        -168.9  240
M4       186.1         -25.1  2414       152.6         -55.7  240
M5       157.4          32.6  2414       173.1        -157.3  240
M6       262.3         -10.3  2414       339.5        -336.8  240

## 4. Skill by model and test year

The same two measures for each test year, first over all valid days and then over the lowest days. Each table has the mean absolute error on the left and the mean error on the right, one column per model.

In [4]:
by_year = evaluate.daily_skill(errors, ["year", "model"])
by_year.to_csv(checks / "daily_skill_by_year.csv")


def year_by_model(day_set: str) -> pd.DataFrame:
    """Mean absolute error and mean error by test year and model, for one set of days."""
    rows = by_year.xs(day_set, level="day_set")
    measures = {name: rows[name].unstack("model") for name in ("mae_mw", "mean_error_mw")}
    return pd.concat(measures, axis=1)


print("All valid days (MW)")
year_by_model("all days")

All valid days (MW)


mae_mw                         mean_error_mw                           
model     M2    M3    M4    M5    M6            M2     M3     M4    M5     M6
year                                                                         
2019   200.9 204.7 139.8 119.6 165.6        -118.0 -120.0 -101.0  75.0  -10.5
2020   297.0 282.1 226.3 137.3 231.8        -216.4 -182.2 -193.6 -12.0 -123.2
2021   332.5 272.1 200.2 158.8 285.0        -268.2 -164.1 -144.2  28.6 -147.6
2022   235.2 218.6 186.6 166.4 210.2         -77.9   80.8   71.3 117.5   77.6
2023   295.0 269.3 169.5 187.9 279.9        -179.3  -31.1  -35.8 -44.1  -69.7
2024   277.4 279.6 179.6 191.1 314.7         -10.2  148.8   86.0  39.9  -57.6
2025   285.1 298.0 201.5 144.0 353.4          59.2  175.9  138.9  22.6  238.8

In [5]:
print("Lowest days of each year (MW)")
year_by_model("lowest days")

Lowest days of each year (MW)


mae_mw                         mean_error_mw                            
model     M2    M3    M4    M5    M6            M2     M3     M4     M5     M6
year                                                                          
2019   222.5 219.7 125.6  90.9 198.7        -222.5 -219.7 -120.1  -90.8 -198.7
2020   293.2 238.2 193.5 159.8 287.1        -293.2 -236.6 -190.0 -159.0 -287.1
2021   392.0 258.8 166.0 177.4 600.6        -392.0 -253.4 -158.9 -176.8 -600.6
2022   217.8 176.1 207.5  92.2 160.3        -196.2   -5.4  126.0   -5.3 -154.7
2023   317.9 179.7  95.4 352.9 505.1        -317.9 -159.5  -48.0 -352.9 -505.1
2024   277.7 161.0 131.7 183.0 403.5        -271.1 -105.1   -0.4 -175.5 -403.5
2025   329.3 246.4 149.1 156.5 242.9        -316.4 -202.9   -4.6 -144.2 -230.4

## 5. The three pairs

Each pair compares two models on mean absolute error: both errors, their difference, and the change in per cent. A negative change means the first model of the pair misses by less.

- **M3 against M2** swaps time for PV capacity. Neither uses the weather, so this is a one-year-ahead test.
- **M4 against M3** adds weather in the linear model, and **M5 against M6** adds it in XGBoost. Both weather models are given the actual weather of the day, so these two pairs show how much weather explains, not how much it helps a year ahead.

The first table is over all test years. The second gives the change in per cent year by year, with the number of years (out of seven) in which the first model misses by less.

In [6]:
pairs = evaluate.compare_daily(errors, PAIRS)
shown = ["model_mae_mw", "reference_mae_mw", "difference_mw", "change_pct"]
pairs[shown]

model_mae_mw  reference_mae_mw  difference_mw  change_pct
comparison                                    day_set                                                               
Impact of PV in place of time (M3 against M2) all days            260.3             274.1          -13.8        -5.0
                                              lowest days         211.7             292.4          -80.8       -27.6
Impact of weather, linear (M4 against M3)     all days            186.1             260.3          -74.2       -28.5
                                              lowest days         152.6             211.7          -59.0       -27.9
Impact of weather, XGBoost (M5 against M6)    all days            157.4             262.3         -105.0       -40.0
                                              lowest days         173.1             339.5         -166.4       -49.0

In [7]:
pairs_by_year = evaluate.compare_daily(errors, PAIRS, by=("year",))
pairs_by_year.to_csv(checks / "daily_pairs_by_year.csv")
# Unstacking sorts the rows by name; they go back in the order of the table above.
change = pairs_by_year["change_pct"].unstack("year").reindex(pairs.index)
change.assign(years_better=(change < 0).sum(axis=1))

year                                                       2019  2020  2021  2022  2023  2024  2025  years_better
comparison                                    day_set                                                            
Impact of PV in place of time (M3 against M2) all days      1.9  -5.0 -18.1  -7.0  -8.7   0.8   4.5             4
                                              lowest days  -1.3 -18.7 -34.0 -19.2 -43.5 -42.0 -25.2             7
Impact of weather, linear (M4 against M3)     all days    -31.7 -19.8 -26.4 -14.7 -37.1 -35.8 -32.4             7
                                              lowest days -42.8 -18.8 -35.9  17.9 -46.9 -18.2 -39.5             6
Impact of weather, XGBoost (M5 against M6)    all days    -27.8 -40.8 -44.3 -20.9 -32.9 -39.3 -59.3             7
                                              lowest days -54.2 -44.3 -70.5 -42.5 -30.1 -54.6 -35.6             7

## 6. Table II

The totals of sections 3 and 5 in one table: a row for each model, then a row for each pair. The table has the shape of the paper's table: one column for all valid days and one for the lowest days. A model row gives its mean absolute error in MW. A pair row gives the change in mean absolute error in per cent, and is named for what the pair changes, with the two models in brackets. The `unit` column says which of the two a row holds. The number of days and the mean error are not in the table; they are in the check of section 3. The differences in MW are in section 5.

In [8]:
table2 = evaluate.daily_table(errors, PAIRS)
table2.to_csv(tables / "table2.csv")
table2.T

row,M2,M3,M4,M5,M6,Impact of PV in place of time (M3 against M2),"Impact of weather, linear (M4 against M3)","Impact of weather, XGBoost (M5 against M6)"
unit,MW,MW,MW,MW,MW,%,%,%
mae_all_days,274.1,260.3,186.1,157.4,262.3,-5.0,-28.5,-40.0
mae_lowest_days,292.4,211.7,152.6,173.1,339.5,-27.6,-27.9,-49.0


## 7. Result

The three pairs in one place: the mean absolute error of each model of the pair and the change, over all valid days and over the lowest days, with the number of test years in which the first model misses by less.

In [9]:
years_better = (change < 0).sum(axis=1).rename("years_better_of_7")
result = pairs[["model_mae_mw", "reference_mae_mw", "change_pct"]].join(years_better)
valid, lowest = days.loc["all years", ["valid_days", "lowest_days"]]
first, last = errors["year"].min(), errors["year"].max()
print(
    f"Daytime low, test years {first} to {last}: "
    f"{valid:,} valid days, of which {lowest:,} are lowest days"
)
print(result.round(1).to_string())

Daytime low, test years 2019 to 2025: 2,414 valid days, of which 240 are lowest days
                                                           model_mae_mw  reference_mae_mw  change_pct  years_better_of_7
comparison                                    day_set                                                                   
Impact of PV in place of time (M3 against M2) all days            260.3             274.1        -5.0                  4
                                              lowest days         211.7             292.4       -27.6                  7
Impact of weather, linear (M4 against M3)     all days            186.1             260.3       -28.5                  7
                                              lowest days         152.6             211.7       -27.9                  6
Impact of weather, XGBoost (M5 against M6)    all days            157.4             262.3       -40.0                  7
                                              lowest days         17